1.Problem statement.

In [ ]:
"""
Customer Churn Prediction using Machine Learning

Goal:
Predict whether a telecom customer will churn or not.
This is a binary classification problem.
"""

2.Import Libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc

3.Load Dataset.

In [ ]:
df = pd.read_csv('Projects/Telco Customer Churn Prediction/Telco_Customer_Churn.csv')

display(df.head())

4.EDA

In [ ]:
print("=== DATA INFO ===")
df.info()

print("\n=== DESCRIPTIVE STATS ===")
display(df.describe())

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

5.Data Cleaning

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("Missing after conversion:")
print(df.isnull().sum())

df.dropna(subset=['TotalCharges'], inplace=True)
df.reset_index(drop=True, inplace=True)

print("\nAfter cleaning:")
print(df.isnull().sum())

6.Data Visualization

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Churn", hue="Churn", palette="coolwarm", legend=False)
plt.title("Churn Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Churn", y="tenure", hue="Churn", palette="viridis", legend=False)
plt.title("Tenure vs Churn")
plt.grid()
plt.show()

7.Data processing.

In [ ]:
# Drop unnecessary column
df.drop('customerID', axis=1, inplace=True)

# Convert target
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

# Split features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

8.Feature scaling.

In [ ]:
scaler = StandardScaler()

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train = X_train.copy()
X_test = X_test.copy()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

9. Model Training & Evaluation

In [ ]:
results = []

def evaluate_model(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    })

evaluate_model("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42))
evaluate_model("Random Forest", RandomForestClassifier(random_state=42))
evaluate_model("Decision Tree", DecisionTreeClassifier(random_state=42))
evaluate_model("KNN", KNeighborsClassifier())
evaluate_model("SVM", SVC(probability=True, random_state=42))

10.Model Comparison.

In [ ]:
results_df = pd.DataFrame(results)
display(results_df.sort_values(by='F1 Score', ascending=False))

11.ROC-AUC.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True, random_state=42)
}

plt.figure(figsize=(10,8))

roc_scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:,1]
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    roc_scores[name] = roc_auc
    
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

plt.plot([0,1], [0,1], 'k--')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.legend()
plt.grid()
plt.show()

best_model = max(roc_scores, key=roc_scores.get)
print("Best Model (AUC):", best_model)

12.Feature Importance (Random Forest)

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

importance = pd.Series(rf.feature_importances_, index=X.columns)
display(importance.sort_values(ascending=False).head(10))

In [ ]:
"""
Key Insights:

- Low tenure customers are more likely to churn
- Month-to-month contracts have higher churn
- Higher monthly charges increase churn risk
- Lack of services leads to churn

Best Model:
Logistic Regression (based on ROC-AUC)

Conclusion:
Model can help identify high-risk customers and improve retention strategies.
"""